# Fine-tune a procurement extractor — Colab run (synthetic data)

Trains **Qwen2.5-7B-Instruct** (QLoRA, 4-bit) on 2,000 synthetic quotations
to output strict JSON with anomaly flags, then compares it against the base
model on 100 held-out quotes.

**Setup once:** Runtime → Change runtime type → **T4 GPU**.
**Then:** Run all cells (Ctrl+F9). Cell 3 will ask you to upload
`procurement-ft-trial.zip`. Expect ~45-70 minutes total. All data is
fictional — nothing real leaves your machine.

In [ ]:
import subprocess, sys, time
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:400])

In [ ]:
print("Installing Soup training stack... (one-time, ~2-3 min)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
  "soup-cli[train]", "accelerate", "bitsandbytes"], check=True)
print("Installed. Soup version:")
subprocess.run([sys.executable, "-m", "soup", "--version" if False else "--help"],
               capture_output=True, text=True)

In [ ]:
# Data comes straight from the public repo (all synthetic) -- no manual upload.
import os, subprocess, sys
REPO = "https://github.com/rubinagentagi-tech/procurement-extractor-finetune.git"
if not os.path.isdir("procurement-extractor-finetune"):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO], capture_output=True, text=True)
    print(r.stdout[-400:]); print(r.stderr[-400:])
os.chdir("procurement-extractor-finetune")
print("Files:", sorted(f for f in os.listdir(".") if not f.startswith(".")))
print("Data rows:", sum(1 for _ in open("data/train.jsonl")))


In [ ]:
import shutil, os
# 1-epoch config for the free tier (2-epoch config lives in soup.yaml for paid GPUs)
shutil.copy("colab-soup.yaml", "soup.yaml")
print("Using config:")
print(open("soup.yaml").read())

In [ ]:
print("Starting training...\n")
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "soup", "train", "-c", "soup.yaml"],
                   capture_output=True, text=True)
print(r.stdout[-3500:])
if r.returncode != 0:
    print("TRAIN STDERR (tail):")
    print(r.stderr[-3000:])
print("EXIT:", r.returncode)

In [ ]:
import os
# show the loss curve if logged
logd = "output"
if os.path.isdir(logd):
    for f in sorted(os.listdir(logd))[:20]: print(" ", f)
else:
    print("no output dir")

In [ ]:
print("Evaluating baseline vs fine-tuned (100 quotes)...")
import subprocess, sys
r = subprocess.run([sys.executable, "eval_check.py",
                    "--base", "Qwen/Qwen2.5-7B-Instruct",
                    "--adapter", "./output"],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-2000:])

## Save your proof
The next cell zips the adapter + config + eval log and downloads it to your
computer. Keep it — it is your *"I trained a model"* artifact (adapter
~150 MB, merges back into Qwen2.5-7B-Instruct with `soup merge`).

In [ ]:
import zipfile, os
with zipfile.ZipFile("colab-trial-results.zip", "w") as z:
    for root, _, fs in os.walk("output"):
        for f in fs:
            p = os.path.join(root, f)
            z.write(p, p)
    for f in ("soup.yaml", "colab-soup.yaml"):
        if os.path.exists(f): z.write(f)
from google.colab import files
files.download("colab-trial-results.zip")
print("Downloaded: colab-trial-results.zip (adapter + config + logs)")